In [1]:
import pandas as pd
import numpy as np
import os

Prior Notebook: `\Scripts and Notebooks\Cleaned Data\2023-03-10 Clean Names`

### 2023-03-10 | Further Cleaning of Author Rows

Want to check for duplicates article titles, and if there is a duplicate, only keep the citation from the more-cited query author.

In [2]:
def get_unique_auth(df):
    
    author_list = np.concatenate(df.loc[:, 'first_auth':'third_auth'].values)
    auth_ser = pd.Series(author_list)
    
    og_names = auth_ser.unique().shape[0]
    print(f"There are {og_names} unique author names")

In [3]:
# import papers_df
clean_path = r'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = '2023-03-10 Clean Names_paper_df.csv'
papers_df = pd.read_csv(os.path.join(clean_path, clean_file))

In [5]:
papers_df.head(3)

,Cites,Title,Year,Source,GSRank,ECC,CitesPerYear,CitesPerAuthor,AuthorCount,query_id,first_auth,second_auth,third_auth,ArticleType,SJR,Hindex,SourceCountry,SourceRegion,query_author
0,4186,low trap-state density and long carrier diffus...,2015.0,science,1,4186,523.25,523,8,0,d shi,v adinolfi,r comin,journal,14.589,1229,United States,Northern America,eh sargent
1,1951,efficient and stable solution-processed planar...,2017.0,science,4,1951,325.17,279,7,0,h tan,a jain,o voznyy,journal,14.589,1229,United States,Northern America,eh sargent
2,1756,homogeneously dispersed multimetal oxygen-evol...,2016.0,science,7,1756,250.86,251,7,0,bo zhang,x zheng,o voznyy,journal,14.589,1229,United States,Northern America,eh sargent


In [6]:
get_unique_auth(papers_df)

There are 16250 unique author names


Let's look to see if there are any null values.

In [7]:
num = papers_df.isna().sum().sum()
print(f'There are {num} null values in the data set')
get_unique_auth(papers_df)

There are 814 null values in the data set
There are 16250 unique author names


In [8]:
papers_df.isna().sum()

Cites               0
Title               2
Year                0
Source              0
GSRank              0
ECC                 0
CitesPerYear        0
CitesPerAuthor      0
AuthorCount         0
query_id            0
first_auth          0
second_auth         0
third_auth        812
ArticleType         0
SJR                 0
Hindex              0
SourceCountry       0
SourceRegion        0
query_author        0
dtype: int64

There is one null value in `Title`. Let's just remove it.

In [9]:
papers_df = papers_df.dropna(subset = 'Title')

In [10]:
num = papers_df.isna().sum().sum()
print(f'There are {num} null values in the data set')
get_unique_auth(papers_df)

There are 812 null values in the data set
There are 16250 unique author names


In [11]:
papers_df.isna().sum()/papers_df.isna().count()

Cites             0.000000
Title             0.000000
Year              0.000000
Source            0.000000
GSRank            0.000000
ECC               0.000000
CitesPerYear      0.000000
CitesPerAuthor    0.000000
AuthorCount       0.000000
query_id          0.000000
first_auth        0.000000
second_auth       0.000000
third_auth        0.036044
ArticleType       0.000000
SJR               0.000000
Hindex            0.000000
SourceCountry     0.000000
SourceRegion      0.000000
query_author      0.000000
dtype: float64

This helped decrease the number of unique authors by about 24.

The remaining null values are for `third_auth`. Initially, I wanted to keep the rows with null `third_auth` columns, but since it only contains about 4% of the data, let's remove those rows for completeness.

In [12]:
papers_df = papers_df.dropna(subset = 'third_auth')

In [13]:
num = papers_df.isna().sum().sum()
print(f'There are {num} null values in the data set')
get_unique_auth(papers_df)

There are 0 null values in the data set
There are 16150 unique author names


Let's remove the duplicated citations.

First, let's remove some special characters from the titles incase that increases the number of duplicate citations.

In [14]:
import re

def replace_char(row):
    print(row['Title'])
    return re.sub(r'[\!\:\-\,]', '', row['Title'])
    
papers_df = papers_df.assign(Title = papers_df.apply(replace_char, axis = 1))

low trap-state density and long carrier diffusion in organolead trihalide perovskite single crystals
efficient and stable solution-processed planar perovskite solar cells via contact passivation
homogeneously dispersed multimetal oxygen-evolving catalysts
co2 electroreduction to ethylene via hydroxide-mediated copper catalysis at an abrupt interface
what would it take for renewably powered electrosynthesis to displace petrochemical processes?
challenges for commercializing perovskite solar cells
building devices from colloidal quantum dots
colloidal quantum-dot photodetectors exploiting multiexciton generation
co2 electrolysis to multicarbon products at activities greater than 1 a cm2
efficient tandem solar cells with solution-processed perovskite on textured crystalline silicon
thermal unequilibrium of strained black cspbi3 thin films
semiconductor quantum dots: technological progress and future challenges
co2 electrolysis to multicarbon products in strong acid
chloride-mediated selec

how to quantify the efficiency potential of neat perovskite films: perovskite semiconductors with an implied efficiency exceeding 28%
how to quantify the efficiency potential of neat perovskite films: perovskite semiconductors with an implied efficiency exceeding 28%
ion exchange lithography: localized ion exchange reactions for spatial patterning of perovskite semiconductors and insulators
memory seeds enable high structural phase purity in 2d perovskite films for highefficiency devices
quantifying anionic diffusion in 2d halide perovskite lateral heterostructures
defect activity in lead halide perovskites
deepblue perovskite singlemode lasing through efficient vaporassisted chlorination
controlled crystal plane orientations in the zno transport layer enable highresponsivity, lowdarkcurrent infrared photodetectors
solventassisted kinetic trapping in quaternary perovskites
multidimensional perovskites for high detectivity photodiodes
from heterostructures to solidsolutions: structural 

In [15]:
num = papers_df.duplicated('Title').sum()
print(f'There are {num} duplicated citations')
print('Thats {:.2%} of the citations'.format(num/papers_df.shape[0]))

There are 5329 duplicated citations
Thats 24.54% of the citations


Let's drop the duplicated papers.

In [16]:
papers_df = papers_df.drop_duplicates('Title')

In [17]:
num = papers_df.duplicated('Title').sum()
print(f'There are {num} duplicated citations')
print('Thats {:.2%} of the citations'.format(num/papers_df.shape[0]))

There are 0 duplicated citations
Thats 0.00% of the citations


We can also check if there are duplicated citations with the same `Year` `Cites` `query_ids` and authors

In [18]:
num = papers_df.duplicated(subset = ['Year','Cites','first_auth', 'second_auth', 'third_auth', 'Source', 'query_id']).sum()
print(f'There are {num} duplicated citations')
print('Thats {:.2%} of the citations'.format(num/papers_df.shape[0]))

There are 5 duplicated citations
Thats 0.03% of the citations


Let's drop those duplicates.

In [19]:
papers_df = papers_df.drop_duplicates(subset = ['Year','Cites','first_auth', 'second_auth', 'third_auth', 'Source', 'query_id'])

In [20]:
num = papers_df.duplicated(subset = ['Year','Cites','first_auth', 'second_auth', 'third_auth', 'Source', 'query_id']).sum()
print(f'There are {num} duplicated citations')
print('Thats {:.2%} of the citations'.format(num/papers_df.shape[0]))

There are 0 duplicated citations
Thats 0.00% of the citations


Another thing we can check for is if there are additional duplicated papers based on the author list, that is, we can check if the titles in the `Title` column that are formatted differently.

In [21]:
# Find where a group of author names have more than one citation
def group_papers(group):
    if group['Title'].count() > 1:
        print(group['Title'])
        return  group['Title'].count()
    else:
        return 0

j = 0 
for group_name, group in papers_df.groupby(['first_auth', 'second_auth', 'third_auth']):
    i = group_papers(group)
    j += i

3602    investigation of carriercarrier scattering eff...
3638    hole barrier height reduction in inverted quan...
8604    vacancy formation and oxidation characteristic...
Name: Title, dtype: object
17236    effects of 5ammonium valeric acid iodide as ad...
17240    reduction of grain boundary resistance of la0....
Name: Title, dtype: object
723     steric engineering of metalhalide perovskites ...
3498    photovoltaic effects on the organic ambipolar ...
5918    impedance spectroscopy characterisation of hig...
6116    strong excitonphoton coupling and lasing behav...
6123    twodimensional platinum diselenide waveguidein...
Name: Title, dtype: object
20798    temperature and frequency dependent dielectric...
20872    thiophene and naphtho [1 2c 5 6c] bis [1 2 5] ...
Name: Title, dtype: object
6310    inside cover a roomtemperature verweytype tran...
6340    sitespecific synthesis and in situ immobilizat...
Name: Title, dtype: object
6327    a titanium metalorganic framework with vi

884      short contacts between chains enhancing lumine...
7470     selfassembly and bonding of alkanephosphonic a...
13057    17% eficient thinfilm silicon solar cell by li...
Name: Title, dtype: object
11514    spray coated highconductivity pedot pss transp...
11556    beyond quantum confinement excitonic nonlocali...
Name: Title, dtype: object
12399    an inverted bii3/pcbm binary quasibulk heteroj...
12440    molecular dynamics study of the interactions o...
Name: Title, dtype: object
11977    determination of depolarization temperature of...
12029    optical nonlinearities of glass doped with pbs...
Name: Title, dtype: object
11968    conducting grain boundaries in the highdielect...
12020    metalinsulator transition in nd1x eux nio3 pro...
Name: Title, dtype: object
4250    hydrophobic organic hole transporters for impr...
4254    polarization screening mechanisms at la0.7sr0....
Name: Title, dtype: object
4260    bulk heterojunction perovskite solar cells inc...
4262    efficie

16114    energy transfer in a laddertype methylpoly (pa...
16115    offcentre motion in doped cubic oxides a gener...
Name: Title, dtype: object
16448    a k2nif4type la2li0. 5al0. 5o4 catalyst for th...
16469    cover feature effects of allorganic interlayer...
Name: Title, dtype: object
18179    influence of impurities on the luminescence of...
18217    nickel (ii)mediated cyanamidepyrazole coupling...
Name: Title, dtype: object
1249     colloidal atomic layer deposition with station...
10934    the effect of oxygen vacancy and spinel phase ...
Name: Title, dtype: object
10368    routes toward longterm stability of mixedhalid...
18745    ferroelectric and magnetoelectric behaviors of...
Name: Title, dtype: object
258     integrating graphene into semiconductor fabric...
3766    an insight into the charge carriers transport ...
Name: Title, dtype: object
3767    thermoelectric coolers for highpowerdensity 3d...
9228    understanding and controlling morphology evolu...
Name: Title, dty

13521    direct growth of crystalline triazinebased gra...
13581    novel dirhodium coordination polymers the impa...
Name: Title, dtype: object
15200    xray powder diffraction results for the phase ...
19771    comparative dielectric and ferroelectric chara...
Name: Title, dtype: object
12726    complementary bulk and surface passivations fo...
12751    single molecule spectroscopy of redand greenem...
Name: Title, dtype: object
987      visiblelight copper nanocluster catalysis for ...
13063    electrical properties of perovskite solar cell...
14578    electrode work function reduction by polyethyl...
14606    graphene oxide inserted poly(nvinylcarbazole)/...
Name: Title, dtype: object
1495    dual wavelength electroluminescence from cdse/...
1589    diameterdependent optical absorption and excit...
Name: Title, dtype: object
9951     charge carrier photogeneration and recombinati...
9953     electronic processes in polyaniline films phot...
10094    optical properties of gasca(oh) 

In [22]:
print('Total Papers: ', papers_df.shape[0])
print('Total suspicious paper titles: ', j)

Total Papers:  16382
Total suspicious paper titles:  7425


It looks like a couple of these repeats are citations for the supporting information. Let's remove citations with *supporting information* in the title.

In [23]:
for i in papers_df.index:
    title = papers_df.loc[i, 'Title']
    if np.char.find(title, 'supporting information') != -1 \
        or np.char.find(title, 'reply to the comment') != -1 \
        or np.char.find(title, '{sub') != -1\
        or np.char.find(title, 'correction') != -1\
        or np.char.find(title, 'erratum') != -1:
        
        print('Dropping the paper:')
        print(title)
        papers_df = papers_df.drop(i)

Dropping the paper:
erratum what would it take for renewably powered electrosynthesis to displace petrochemical processes?(science
Dropping the paper:
corrections &amendments
Dropping the paper:
erratum a general phasetransfer protocol for metal ions and its application in nanocrystal synthesis (nature materials (2009) 8 (683689))
Dropping the paper:
erratum corrigendum beating the thermodynamic limit with photoactivation of ndoping in organic semiconductors.
Dropping the paper:
author correction thermally activated delayed fluorescence (tadf) organic molecules for efficient xray scintillation and imaging
Dropping the paper:
author correction roomtemperature superfluorescence in hybrid perovskites and its origins
Dropping the paper:
publisher correction copper adparticle enabled selective electrosynthesis of npropanol
Dropping the paper:
author correction chargegenerating midgap trap states define the thermodynamic limit of organic photovoltaic devices
Dropping the paper:
author correc

Dropping the paper:
high pressure neutron diffraction study of the magnetoresistive 1222type ruthenocuprate rusr {sub 2} nd {sub 0.9} y {sub 0.2} ce {sub 0.9} cu {sub 2} o {sub 10}
Dropping the paper:
a dft study on structural electronic mechanical and thermodynamic properties of 5felectron system baamo {sub 3}
Dropping the paper:
studies of the magnetocaloric and electrical properties of la {sub 0.7} er {sub 0.05} sr {sub 0.15} ca {sub 0.1} mn {sub 1 x} in {sub x} o {sub 3}(0 x 0.30)
Dropping the paper:
modeling the magnetocaloric effect of la {sub 0.67} pb {sub 0.33} mno {sub 3} by the meanfield theory
Dropping the paper:
structural infrared magnetic and electrical properties of ni {sub 0.6} cd {sub 0.2} cu {sub 0.2} fe {sub 2} o {sub 4} ferrites synthesized using solgel method under
Dropping the paper:
microstructural magnetic magnetocaloric and electrical properties of ni {sub 0.4} mg {sub 0.3} cu {sub 0.3} fe {sub 2} o {sub 4} ferrite prepared using solgel method
Dropping the pape

In [24]:
# Find where a group of author names have more than one citation
def group_papers(group):
    if group['Title'].count() > 1:
        print(group['Title'])
        return group['Title'].count()
    else:
        return 0

j = 0 
for group_name, group in papers_df.groupby(['first_auth', 'second_auth', 'third_auth']):
    i = group_papers(group)
    j += i

3602    investigation of carriercarrier scattering eff...
3638    hole barrier height reduction in inverted quan...
8604    vacancy formation and oxidation characteristic...
Name: Title, dtype: object
17236    effects of 5ammonium valeric acid iodide as ad...
17240    reduction of grain boundary resistance of la0....
Name: Title, dtype: object
723     steric engineering of metalhalide perovskites ...
3498    photovoltaic effects on the organic ambipolar ...
5918    impedance spectroscopy characterisation of hig...
6116    strong excitonphoton coupling and lasing behav...
6123    twodimensional platinum diselenide waveguidein...
Name: Title, dtype: object
20798    temperature and frequency dependent dielectric...
20872    thiophene and naphtho [1 2c 5 6c] bis [1 2 5] ...
Name: Title, dtype: object
6310    inside cover a roomtemperature verweytype tran...
6340    sitespecific synthesis and in situ immobilizat...
Name: Title, dtype: object
6327    a titanium metalorganic framework with vi

6954    metal halide perovskite and phosphorus doped g...
7010    the molybdenum oxide interface limits the high...
Name: Title, dtype: object
22392    removal of chromate anions and immobilization ...
22396    physicalchemical characterization of metal hyd...
Name: Title, dtype: object
17508    structural stability and properties of marokit...
17556    cooperative effect of anion and mole ratio on ...
Name: Title, dtype: object
17487    optical and acoustic metamaterials superlens n...
17560    influence of anion and mole ratio on the coord...
Name: Title, dtype: object
13941    electrophoretic deposited oxide thin films as ...
14011    porous silicon nanoparticles as a nanophotocat...
Name: Title, dtype: object
14416    tuning the spectral response of ultraviolet or...
14461    preparation and electrical properties of bi3. ...
Name: Title, dtype: object
14432    unexpectedly high fieldeffect mobility of a so...
14477    porositymoderated ultrafast electron transport...
Name: Title, d

3049    shapepure nearly monodispersed cspbbr3 nanocub...
3187    efficient schottkyquantumdot photovoltaics the...
Name: Title, dtype: object
21477    gga based study on electronic structure and th...
21553    synthesis morphology and optical properties of...
Name: Title, dtype: object
922    dna clutch probes for circulating tumor dna an...
926    niremitting colloidal quantum dots having 26% ...
Name: Title, dtype: object
14216    effect of solvent and additives on the opencir...
19351    unexpected rotamerism at the origin of a chess...
Name: Title, dtype: object
13195    ion bombardmentinduced enhancement of the prop...
13205    synthesis and properties of polyfluorene conta...
Name: Title, dtype: object
5035     extended intermolecular interactions governing...
14575    coating of vertically aligned carbon nanotubes...
14603    shuntblocking layers for semitransparent perov...
20567    the maragingsteel blades of the virgo super at...
Name: Title, dtype: object
5001     polarizat

5844     mitigating detrimental effect of selfdoping ne...
14961    stabilizing p3type oxides as cathodes for high...
14977    determining the role of fedoping on promoting ...
Name: Title, dtype: object
8385    amorphous iron oxyhydroxide nanosheets synthes...
8453    size dependent resonance energy transfer betwe...
Name: Title, dtype: object
8427    from generation to extraction a timeresolved i...
8495    ultrafast relaxation dynamics of conjugated po...
Name: Title, dtype: object
4854     structureproperty relationships of the 10h hex...
14950    unfolding the influence of metal doping on pro...
14966    cu (i)/cu (ii) creutztaube mixedvalence 2d coo...
Name: Title, dtype: object
11395    dyesensitized solar cells based on semiconduct...
11460        modelling of hot carrier solar cell absorbers
Name: Title, dtype: object
12643    leadfree halide double perovskites toward stab...
12666    diagnosis of electrically active defects in ch...
Name: Title, dtype: object
12642    light d

11218    can trihalide lead perovskites support continu...
11292    narrowband monolithic perovskiteperovskite tan...
Name: Title, dtype: object
16162    effects of pr substitution on electrical prope...
16164    influence of mo backcontact oxidation on prope...
16172    tandem photovoltaicphotoelectrochemical gaas/i...
16189    constraints in postsynthesis ligand exchange f...
16190    polymeric stabilization of hybrid nanocomposit...
16199    insitu synthesis of thiophenebased multifuncti...
16201    syntheses and properties of electroluminescent...
16226    stability of organicinorganic hybrid perovskit...
Name: Title, dtype: object
12631    stabilizing wide bandgap triplehalide perovski...
12654    analysis of nonquarterwave grating by a modifi...
Name: Title, dtype: object
12739    optical control over photoconductivity in poly...
12764    local symmetry change in  at  microscopic insight
Name: Title, dtype: object
2870    electronic and morphological inhomogeneities i...
2959    

In [25]:
print('Total Papers: ', papers_df.shape[0])
print('Total suspicious paper titles: ', j)

Total Papers:  16304
Total suspicious paper titles:  7363


It also looks like some of the papers have titles written in English and then a repeated title written in a different language (most likely German). Let's try to remove the duplicated title names that aren't in English.

In [27]:
from langdetect import detect
import re

# Find where a group of author names have more than one citation
def group_papers(group):
    if group['Title'].count() > 1:
        for i in group['Title'].values:
            string = ''.join(re.findall(r"[a-zA-Z ]", i))
            
            # Looks for titles in German, searching for 'non-English' was giving too many English results
            # Specifically ignoring 'quantum diffusion of deuterium in gaas zn' since its in english but it determined to be german
            if detect(string) == 'de' and string != 'quantum diffusion of deuterium in gaas zn':
                return True
    else:
        return False
                
results = papers_df.groupby(['first_auth', 'second_auth', 'third_auth']).apply(group_papers)

In [29]:
print('Here are the authors where there a duplicate paper names in German')
results[results == True]

Here are the authors where there a duplicate paper names in German


first_auth   second_auth        third_auth     
a giuri      r munir            a listorti         True
a trueba     p garcafernndez    jm garcialastra    True
ja aramburu  p garciafernandez  nr mathiesen       True
k zhao       h hu               e spada            True
mc tang      d barrit           r munir            True
             y fan              d barrit           True
sj lee       mc tang            r munir            True
dtype: object

In [30]:
for auth in results[results == True].index:
    f, s, t = auth
    test_df = papers_df[(papers_df['first_auth'] == f) & (papers_df['second_auth'] == s) & (papers_df['third_auth'] == t)]
    for i in test_df.index:
        if detect(test_df.loc[i, 'Title']) == 'de':
            print('Dropping the paper:')
            print(test_df.loc[i, 'Title'])
            papers_df = papers_df.drop(i)

Dropping the paper:
perowskitoxidelektroden zur leistungsstarken photoelektrochemischen wasserspaltung
Dropping the paper:
visualisierung der phasensegregation in gemischthalogenidperowskiteinkristallen
Dropping the paper:
von vorluferpulvern zu cspbx3perowskitnanodrhten eintopfreaktion wachstumsmechanismus und gerichtete selbstassemblierung
Dropping the paper:
perowskitsolarzellen atomare ebene schichtqualitt und leistungsfhigkeit der zellen
Dropping the paper:
quantisierte aufladung von ligandenfreien clustern in einer ionischen flssigkeit
Dropping the paper:
fluoridchemie in zinnhalogenidperowskiten
Dropping the paper:
innenrcktitelbild fluoridchemie in zinnhalogenidperowskiten (angew. chem. 39/2021)


In [31]:
# Find where a group of author names have more than one citation
def group_papers(group):
    if group['Title'].count() > 1:
        print(group['Title'].str.rstrip(' '))
        return group['Title'].count()
    else:
        return 0

j = 0 
for group_name, group in papers_df.groupby(['first_auth', 'second_auth', 'third_auth']):
    i = group_papers(group)
    j += i

3602    investigation of carriercarrier scattering eff...
3638    hole barrier height reduction in inverted quan...
8604    vacancy formation and oxidation characteristic...
Name: Title, dtype: object
17236    effects of 5ammonium valeric acid iodide as ad...
17240    reduction of grain boundary resistance of la0....
Name: Title, dtype: object
723     steric engineering of metalhalide perovskites ...
3498    photovoltaic effects on the organic ambipolar ...
5918    impedance spectroscopy characterisation of hig...
6116    strong excitonphoton coupling and lasing behav...
6123    twodimensional platinum diselenide waveguidein...
Name: Title, dtype: object
20798    temperature and frequency dependent dielectric...
20872    thiophene and naphtho [1 2c 5 6c] bis [1 2 5] ...
Name: Title, dtype: object
6310    inside cover a roomtemperature verweytype tran...
6340    sitespecific synthesis and in situ immobilizat...
Name: Title, dtype: object
6327    a titanium metalorganic framework with vi

12086    transmission electron microscopy of the induce...
12131    sputtering growth and optical properties of [1...
Name: Title, dtype: object
8825    upconversion emission enhancement in yb3+/er3+...
8866    charge carrier mobility of siliconized liquid ...
Name: Title, dtype: object
9829    ferromagnetic interaction between cu ions in t...
9846    semiconductors ii surfaces interfaces microstr...
Name: Title, dtype: object
10218    electronic and excitonic processes in lightemi...
10265    coupling between tilts and charge carriers at ...
Name: Title, dtype: object
13938    amphiphilic acids as coadsorbents of metalfree...
13939    oxide planar pn heterojunction prepared by low...
14009    continuous flow synthesis of citrate capped go...
Name: Title, dtype: object
13931    uvpatternable nanocomposite containing cdse an...
14001    fabrication and performance evaluation of new ...
Name: Title, dtype: object
13207    synthesis and photovoltaic properties of alkox...
13242    nearint

In [32]:
print('Total Papers: ', papers_df.shape[0])
print('Total suspicious paper titles: ', j)

Total Papers:  16297
Total suspicious paper titles:  7350


All the remaining 'suspicious' papers look to be genuinely different from each other. Let's check for null values and the unique author count.

In [33]:
num = papers_df.isna().sum().sum()
print(f'There are {num} null values in the data set')
get_unique_auth(papers_df)

There are 0 null values in the data set
There are 14118 unique author names


It looks like this round of cleaning did not introduce any null values and also removed an additional 33 unique author names.

Let's check to see if we can do any additional cleaning, such as changing data types of columns.

In [34]:
papers_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16297 entries, 0 to 22528
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Cites           16297 non-null  int64  
 1   Title           16297 non-null  object 
 2   Year            16297 non-null  float64
 3   Source          16297 non-null  object 
 4   GSRank          16297 non-null  int64  
 5   ECC             16297 non-null  int64  
 6   CitesPerYear    16297 non-null  float64
 7   CitesPerAuthor  16297 non-null  int64  
 8   AuthorCount     16297 non-null  int64  
 9   query_id        16297 non-null  int64  
 10  first_auth      16297 non-null  object 
 11  second_auth     16297 non-null  object 
 12  third_auth      16297 non-null  object 
 13  ArticleType     16297 non-null  object 
 14  SJR             16297 non-null  float64
 15  Hindex          16297 non-null  int64  
 16  SourceCountry   16297 non-null  object 
 17  SourceRegion    16297 non-null 

Let's change the data type of `Year` from `float64` to `int16`

In [35]:
papers_df['Year'] = papers_df['Year'].astype('int16')

Let's look at the data types of the numeric data types further to see if we can reduce the data types into smaller integer data types.

In [36]:
papers_df.select_dtypes('number').info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16297 entries, 0 to 22528
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Cites           16297 non-null  int64  
 1   Year            16297 non-null  int16  
 2   GSRank          16297 non-null  int64  
 3   ECC             16297 non-null  int64  
 4   CitesPerYear    16297 non-null  float64
 5   CitesPerAuthor  16297 non-null  int64  
 6   AuthorCount     16297 non-null  int64  
 7   query_id        16297 non-null  int64  
 8   SJR             16297 non-null  float64
 9   Hindex          16297 non-null  int64  
dtypes: float64(2), int16(1), int64(7)
memory usage: 1.3 MB


In [37]:
print('Max Values of each columns')
papers_df.select_dtypes('number').max()

Max Values of each columns


Cites             9210.000
Year              2023.000
GSRank             990.000
ECC               9210.000
CitesPerYear       921.000
CitesPerAuthor    1316.000
AuthorCount         12.000
query_id           615.000
SJR                 23.876
Hindex            1276.000
dtype: float64

All the values are small enough that they can be converted into a `int16` data type. Let's change all the data types, besides `CitesPerYear` since that needs to stay a float.

In [38]:
for i in papers_df.select_dtypes('number').columns:
    if i != 'CitesPerYear': 
        papers_df[i] = papers_df[i].astype('int16')
    else:
        papers_df[i] = papers_df[i].astype('float16')

In [39]:
papers_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16297 entries, 0 to 22528
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Cites           16297 non-null  int16  
 1   Title           16297 non-null  object 
 2   Year            16297 non-null  int16  
 3   Source          16297 non-null  object 
 4   GSRank          16297 non-null  int16  
 5   ECC             16297 non-null  int16  
 6   CitesPerYear    16297 non-null  float16
 7   CitesPerAuthor  16297 non-null  int16  
 8   AuthorCount     16297 non-null  int16  
 9   query_id        16297 non-null  int16  
 10  first_auth      16297 non-null  object 
 11  second_auth     16297 non-null  object 
 12  third_auth      16297 non-null  object 
 13  ArticleType     16297 non-null  object 
 14  SJR             16297 non-null  int16  
 15  Hindex          16297 non-null  int16  
 16  SourceCountry   16297 non-null  object 
 17  SourceRegion    16297 non-null 

Let's save this to a csv

In [44]:
clean_path = r'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = '2023-03-10 Additional Author Cleaning_papers_df.csv'
papers_df.to_csv(os.path.join(clean_path, clean_file))

---

Since these data types will be lost when the file is saved and imported as a csv, the module `clean_dataframe` should be used as part of the importing process.

In [41]:
import sys
import os
import pandas as pd
package_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "Modules"))
sys.path.append(package_dir)

from clean_dataframe.preprocessing import dtype16

In [45]:
# import papers_df
clean_path = r'G:\My Drive\BrainStation\BrainStationCapstone\Scripts and Notebooks\Cleaned Data'
clean_file = '2023-03-10 Clean Names_paper_df.csv'
test_df = pd.read_csv(os.path.join(clean_path, clean_file))

Looking at the data types before:

In [46]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 22530 entries, 0 to 22529
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Cites           22530 non-null  int64  
 1   Title           22528 non-null  object 
 2   Year            22530 non-null  float64
 3   Source          22530 non-null  object 
 4   GSRank          22530 non-null  int64  
 5   ECC             22530 non-null  int64  
 6   CitesPerYear    22530 non-null  float64
 7   CitesPerAuthor  22530 non-null  int64  
 8   AuthorCount     22530 non-null  int64  
 9   query_id        22530 non-null  int64  
 10  first_auth      22530 non-null  object 
 11  second_auth     22530 non-null  object 
 12  third_auth      21718 non-null  object 
 13  ArticleType     22530 non-null  object 
 14  SJR             22530 non-null  float64
 15  Hindex          22530 non-null  int64  
 16  SourceCountry   22530 non-null  object 
 17  SourceRegion    22530 non-null 

The data types after using the `dtype16` module:

In [47]:
dtype16(test_df, use_float=True, float_col= 'CitesPerYear').info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 22530 entries, 0 to 22529
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Cites           22530 non-null  int16  
 1   Title           22528 non-null  object 
 2   Year            22530 non-null  int16  
 3   Source          22530 non-null  object 
 4   GSRank          22530 non-null  int16  
 5   ECC             22530 non-null  int16  
 6   CitesPerYear    22530 non-null  float16
 7   CitesPerAuthor  22530 non-null  int16  
 8   AuthorCount     22530 non-null  int16  
 9   query_id        22530 non-null  int16  
 10  first_auth      22530 non-null  object 
 11  second_auth     22530 non-null  object 
 12  third_auth      21718 non-null  object 
 13  ArticleType     22530 non-null  object 
 14  SJR             22530 non-null  int16  
 15  Hindex          22530 non-null  int16  
 16  SourceCountry   22530 non-null  object 
 17  SourceRegion    22530 non-null 

This module works well, and should be used when importing data to reduce the memory usage by about a MB

---